<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 38.9 MB/s  0:00:00


### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

--2026-10-30 18:22:09--  http://songs_dataset.zip/
Resolving songs_dataset.zip (songs_dataset.zip)... failed: Name or service not known.
wget: unable to resolve host address ‘songs_dataset.zip’
--2026-10-30 18:22:09--  https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip [following]
--2026-10-30 18:22:09--  https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.

In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['michael-jackson.txt',
 'dolly-parton.txt',
 'rihanna.txt',
 'joni-mitchell.txt',
 'bjork.txt',
 'lin-manuel-miranda.txt',
 'amy-winehouse.txt',
 'cake.txt',
 'janisjoplin.txt',
 'drake.txt',
 'nicki-minaj.txt',
 'patti-smith.txt',
 'dj-khaled.txt',
 'britney-spears.txt',
 'nursery_rhymes.txt',
 'notorious_big.txt',
 'beatles.txt',
 'lorde.txt',
 'lady-gaga.txt',
 'bob-dylan.txt',
 'disney.txt',
 'johnny-cash.txt',
 'ludacris.txt',
 'bruno-mars.txt',
 'eminem.txt',
 'al-green.txt',
 'bob-marley.txt',
 'notorious-big.txt',
 'kanye-west.txt',
 'missy-elliott.txt',
 'nickelback.txt',
 'adele.txt',
 'leonard-cohen.txt',
 'r-kelly.txt',
 'bruce-springsteen.txt',
 'paul-simon.txt',
 'lil-wayne.txt',
 'dickinson.txt',
 'jimi-hendrix.txt',
 'Kanye_West.txt',
 'bieber.txt',
 'Lil_Wayne.txt',
 'nirvana.txt',
 'blink-182.txt',
 'radiohead.txt',
 'dr-seuss.txt',
 'alicia-keys.txt',
 'prince.txt',
 'kanye.txt']

In [4]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/tmp/ipykernel_4121/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [7]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [10]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [12]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [13]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 113045.25
Loss after epoch 1: 65966.625
Loss after epoch 2: 65935.046875
Loss after epoch 3: 65718.328125
Loss after epoch 4: 63875.03125
Loss after epoch 5: 64160.8125
Loss after epoch 6: 64080.3125
Loss after epoch 7: 64814.96875
Loss after epoch 8: 62632.5
Loss after epoch 9: 60452.875
Loss after epoch 10: 59839.875
Loss after epoch 11: 58883.875
Loss after epoch 12: 57716.125
Loss after epoch 13: 56494.1875
Loss after epoch 14: 55817.5625
Loss after epoch 15: 55843.1875
Loss after epoch 16: 51722.3125
Loss after epoch 17: 49858.125
Loss after epoch 18: 49592.375
Loss after epoch 19: 48960.0


(156986, 287740)

### 4 - Ensayar

In [14]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.8954238295555115),
 ('sleep', 0.8665637969970703),
 ('help', 0.8439353108406067),
 ('cry', 0.8351286053657532),
 ('not', 0.8309641480445862),
 ('try', 0.8276920318603516),
 ('peace', 0.8144835233688354),
 ('little', 0.8140542507171631),
 ('twist', 0.8123894929885864),
 ('seems', 0.8079566359519958)]

In [15]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('shake', -0.2287309318780899),
 ('four', -0.23302660882472992),
 ('five', -0.23746736347675323),
 ('six', -0.23784852027893066),
 ('bang', -0.24833035469055176),
 ('our', -0.2553912103176117),
 ('day', -0.2689830958843231),
 ('going', -0.2692137360572815),
 ('here', -0.2699104845523834),
 ('three', -0.2839003801345825)]

In [16]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9813724756240845),
 ('three', 0.9745777249336243),
 ('six', 0.9710812568664551),
 ('seven', 0.9584368467330933),
 ('two', 0.9517239332199097),
 ('sixty', 0.8990397453308105),
 ('one', 0.795121431350708),
 ('crying', 0.7946274280548096),
 ('us', 0.7740067839622498),
 ("i'm", 0.7508398294448853)]

In [17]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9433997869491577),
 ('buy', 0.9396989345550537),
 ('much', 0.9033164381980896),
 ('just', 0.8509097695350647),
 ('hide', 0.8355392813682556)]

In [18]:
# Ensayar con una palabra que no está en el vocabulario:
# w2v_model.wv.most_similar(negative=["diedaa"])

In [19]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 0.06138305  0.05881333 -0.06370755  0.02444775 -0.20152195 -0.18612184
 -0.15284826  0.45487037 -0.04218006  0.03535983  0.13657492 -0.18520045
 -0.18126617  0.22149472 -0.3038098  -0.2397019   0.07094733 -0.05679122
 -0.05166152 -0.23843256 -0.08530398  0.19564743 -0.07679035  0.03797102
  0.07517121 -0.04826119  0.07379342  0.10396571  0.00738085 -0.22764632
 -0.04567216  0.12937427  0.27785635  0.19387512 -0.13509206  0.20856984
  0.40917465 -0.00387058 -0.10631468 -0.09056927  0.02400546 -0.0800494
  0.13400891  0.08833336 -0.01894625  0.08592407 -0.1590556   0.10259219
  0.14459157 -0.12092865 -0.27919155 -0.04061459  0.11382194  0.3136597
 -0.07410164  0.13976608  0.22791268  0.13209176 -0.01811359  0.09772693
  0.0924959  -0.14871807 -0.16348243 -0.13203168 -0.09834307  0.02714351
  0.16531163  0.26052278 -0.03259142 -0.02894571  0.11621213 -0.0697405
  0.09563474 -0.15276304  0.22071072  0.15996547  0.15890133 -0.04711402
 -0.12555073 -0.03993772 -0.10795346  0.01878977  0.07

In [20]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0),
 ('babe', 0.9085158109664917),
 ('someone', 0.8886135816574097),
 ('need', 0.882800817489624),
 ('nothing', 0.8740264773368835),
 ("didn't", 0.8638384342193604),
 ("there's", 0.8526685833930969),
 ('you', 0.8456694483757019),
 ('feed', 0.8445034623146057),
 ('somebody', 0.8362801671028137)]

In [21]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9085157513618469),
 ('someone', 0.8886136412620544),
 ('need', 0.8828008770942688),
 ('nothing', 0.8740264773368835),
 ("didn't", 0.8638384342193604),
 ("there's", 0.8526685833930969),
 ('you', 0.8456694483757019),
 ('feed', 0.8445034027099609),
 ('somebody', 0.8362802267074585),
 ('buy', 0.8351722359657288)]

### 5 - Visualizar agrupación de vectores

In [22]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [23]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [24]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [25]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado, incluido el sustento de las decisiones que tomaron (como modificación de hiperparámetros)**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

---
## Desarrollo del Desafío 2

Corpus propio: el libro *Drácula* de Bram Stoker (Project Gutenberg, libro 345). Cada frase del libro es un "documento".

### 1. Descargar y preparar el texto

Pasos: quitamos el encabezado y el final de Project Gutenberg, separamos el libro en frases con NLTK (`sent_tokenize`, que entiende abreviaturas como *Mr.* y *Dr.*) y separamos cada frase en palabras con gensim (`simple_preprocess`, que pasa a minúsculas y quita signos y números).

In [26]:
!wget -q -U "Mozilla/5.0" -O dracula.txt https://www.gutenberg.org/ebooks/345.txt.utf-8

In [27]:
import re
import nltk
from nltk.tokenize import sent_tokenize
from gensim.utils import simple_preprocess

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

texto = open('dracula.txt', encoding='utf-8-sig').read()
texto = re.split(r'\*\*\* START OF.*?\*\*\*', texto, maxsplit=1, flags=re.S)[-1]
texto = re.split(r'\*\*\* END OF', texto)[0]
texto = texto.replace('_', ' ')          # el libro marca las itálicas con _guiones_

oraciones = [simple_preprocess(f) for f in sent_tokenize(texto)]
oraciones = [o for o in oraciones if len(o) > 1]

print("Frases:", len(oraciones), "| Palabras:", sum(len(o) for o in oraciones))
print("Ejemplo:", oraciones[500])

Frases: 7405 | Palabras: 154152
Ejemplo: ['what', 'devil', 'or', 'what', 'witch', 'was', 'ever', 'so', 'great', 'as', 'attila', 'whose', 'blood', 'is', 'in', 'these', 'veins', 'he', 'held', 'up', 'his', 'arms']


### 2. Entrenar los embeddings

Decisiones:
- `sg=1` (skip-gram): funciona mejor con pocos datos.
- `min_count=5`: el libro es chico, así que guardamos palabras con pocas apariciones.
- `epochs=50`: con pocos datos conviene dar más vueltas.
- `window=5`, `vector_size=100`, `negative=10`: valores típicos para un corpus chico.

In [28]:
from gensim.models import Word2Vec

w2v = Word2Vec(sentences=oraciones, vector_size=100, window=5, min_count=5,
               sg=1, negative=10, epochs=50, workers=1, seed=42)
print("Palabras en el vocabulario:", len(w2v.wv))

Palabras en el vocabulario: 2381


### 3. Palabras más y menos similares

Elegimos términos importantes del libro: personajes, lugares y temas.

In [29]:
terminos = ['count', 'blood', 'night', 'castle', 'lucy', 'garlic', 'death', 'god']

for t in terminos:
    if t not in w2v.wv:
        print(f"'{t}' no está en el vocabulario")
        continue
    print(f"\n=== {t} ===")
    print("  más similares  :", [(w, round(s, 2)) for w, s in w2v.wv.most_similar(t, topn=8)])
    print("  menos similares:", [w for w, _ in w2v.wv.most_similar(negative=[t], topn=5)])


=== count ===
  más similares  : [('he', 0.44), ('dracula', 0.44), ('escape', 0.41), ('fancy', 0.41), ('unlocked', 0.4), ('key', 0.4), ('maps', 0.4), ('whither', 0.39)]
  menos similares: ['unselfish', 'stand', 'dizzy', 'wrung', 'strangely']

=== blood ===
  más similares  : [('bloom', 0.58), ('stained', 0.51), ('cheeks', 0.51), ('veins', 0.51), ('transfusion', 0.5), ('drop', 0.49), ('trickled', 0.49), ('pallor', 0.47)]
  menos similares: ['searching', 'round', 'somewhere', 'brought', 'let']

=== night ===
  más similares  : [('morrow', 0.52), ('sleep', 0.46), ('last', 0.45), ('hadn', 0.44), ('gain', 0.43), ('wakened', 0.43), ('afternoon', 0.43), ('breakfast', 0.43)]
  menos similares: ['almost', 'positively', 'range', 'under', 'several']

=== castle ===
  más similares  : [('sereth', 0.47), ('dracula', 0.46), ('sisters', 0.46), ('wall', 0.44), ('maps', 0.42), ('neighbourhood', 0.41), ('borgo', 0.41), ('bistritza', 0.41)]
  menos similares: ['died', 'sympathy', 'succeeded', 'weak', 'r

**Interpretación:**
- **blood**: lo asocia con el cuerpo y las transfusiones de Lucy (*veins, transfusion, drop, trickled, cheeks, pallor, stained*). Captura bien el tema central del libro.
- **garlic**: aparecen *flowers, wreath, crucifix, neck*, que son las protecciones contra el vampiro. Muy coherente con la trama.
- **castle**: lo asocia con *dracula* y los lugares de Transilvania (*borgo, bistritza, sereth*) y con las tres *sisters* del castillo.
- **lucy**: *miss, westenra* (su nombre completo), *mother, daughter* y *poor* (como la llaman los demás personajes).
- **god**: expresiones religiosas (*thank, blessed, almighty, goodness*).
- **death**: *life, illness, horror, nightmare*.
- **night** y **count** dan resultados más débiles: *night* se mezcla con palabras de tiempo (*morrow, afternoon, breakfast*), y *count* también es un verbo ("to count"), así que su vector mezcla los dos sentidos.

Las similitudes son bajas (0.4–0.58) porque el corpus es chico (~154 mil palabras). Las palabras "menos similares" no son opuestos reales: son palabras sin relación (*found, note, maybe*). Word2Vec no aprende antónimos, solo qué palabras aparecen en contextos parecidos.

### 4. Visualización en 2D

Reutilizamos `reduce_dimensions` (t-SNE) de arriba. Después sacamos las stopwords (the, and, of...) para ver solo palabras con contenido. `MAX_WORDS` es la cantidad de palabras que se muestran: con pocas no se ven grupos, y con muchas las etiquetas se tapan. Se puede cambiar para probar.

In [30]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

vecs, labels = reduce_dimensions(w2v)
mask = [l not in ENGLISH_STOP_WORDS for l in labels]
vecs, labels = vecs[mask], labels[mask]

MAX_WORDS = 150
fig = px.scatter(x=vecs[:MAX_WORDS, 0], y=vecs[:MAX_WORDS, 1], text=labels[:MAX_WORDS])
fig.update_traces(textposition='top center', marker_size=4)
fig.update_layout(height=750)
fig.show(renderer="colab")

### 5. Grupos de palabras y conclusiones

Grupos que se ven:
- **Nombres de personajes (derecha)**: *van, helsing, dr, seward, harker* juntos arriba, y *mr, quincey, morris, godalming* abajo. El modelo junta las palabras que forman un nombre completo (Dr. Seward, Van Helsing, Quincey Morris), porque siempre aparecen una al lado de la otra.
- **Documentos (derecha)**: *diary, journal, letter* cerca de *london* y *ship*. Tiene sentido porque la novela está escrita como diarios y cartas.
- **Tiempo**: *night, day, morning* forman su propio grupo.
- **Cuerpo y colores (abajo a la izquierda)**: *red, white, eyes, face, head, hand(s), lay*. Son las descripciones del vampiro y de las víctimas (labios rojos, cara pálida).
- **Miedo y muerte (izquierda)**: *fear, terrible, death, sleep, bed, husband, sweet*.
- **Verbos de diálogo (centro arriba)**: *speak, believe, know, love, ask, tell, want, think, hope, shall*. Aparecen sobre todo en las conversaciones.
- **Lugares de la casa (abajo)**: *door, room, window, light*.
- **Trato afectivo (arriba)**: *madam, mina, lucy, poor, dear, god, john, friend, good*. Son formas de dirigirse a alguien en las cartas y diálogos.

**Conclusiones:** aunque el libro es chico, los embeddings separan bien nombres propios, objetos, temas (sangre, protección, religión) y tipos de verbos. Con 150 palabras ya hay etiquetas superpuestas en el centro (palabras muy frecuentes y de uso general), así que no conviene usar más. Los resultados más débiles (*count*, *night*) muestran el límite del modelo: con poco texto, y con palabras que tienen varios significados, los vectores son menos confiables.